# 🇨🇳📊 Валютный Cash-and-Carry на Московской бирже (MOEX): Бэктест 01.01.2023 – 20.08.2026 гг.

**Автор:** Jules / MOEX Quant Trading Desk  
**Начальный капитал:** `1,000,000 рублей`  
**Период анализа:** `01 января 2023 года — 20 августа 2026 года (Сегодня)`  
**Тарифный план:** «Стандартный ФОРТС» (Комиссия брокера 0.45 ₽/контракт через ИТС)  
**Данные:** Только реальные исторические и текущие цены MOEX ISS API (`CNYRUB_TOM` и 15 квартальных фьючерсов `CRH3` — `CRU6`)

---

## 🛠️ Что такое Валютный Cash-and-Carry и как он работает?

**Cash-and-Carry (Кэш-энд-Кэрри)** — это классическая межрыночная арбитражная стратегия, заключающаяся в:
1. **Покупке реального базового актива** на спот-рынке (Валютная секция MOEX: `CNYRUB_TOM`).
2. **Одновременной продаже (шорте) фьючерсного контракта** на этот же актив на срочном рынке FORTS (квартальные фьючерсы `CRH3` — `CRU6`).

### 💡 Механика и Источники Доходности:
- Фьючерс обычно торгуется с премией к споту (**контанго**). Эта премия отражает стоимость финансирования (процентную ставку) за период до экспирации.
- В день экспирации фьючерса его цена **гарантированно сходится с ценой спот-рынка** (согласно спецификации контракта MOEX).
- Фиксируя спред в день входа, арбитражер получает **абсолютно безрисковую доходность в рублях**, независимую от того, куда пойдет курс юаня (вверх или вниз).

### 🛡️ Управление риском бэквордации (Санкции 2024–2026 гг.):
При локальном дефиците юаней фьючерсы временно уходят в **бэквордацию** (цена фьючерса ниже спота).  
**Умное правило алгоритма (Smart Filter):**
- Если рынок находится в контанго (`Контанго > 0.5%`) — алгоритм исполняет **Cash-and-Carry**.
- Если рынок находится в бэквордации — алгоритм размещает капитал в **Денежный рынок MOEX (СВОП / RUSFAR)**, зарабатывая ставку ЦБ РФ.

---

## 💰 Тарифный план «Стандартный ФОРТС» и Залоговое Обеспечение (ГО):
- **Лотность:** 1 лот спота `CNYRUB_TOM` = 1,000 CNY. 1 контракт фьючерса `CR` = 1,000 CNY. Хедж 1:1.
- **Комиссия спот-рынка:** `0.025%` от суммы сделки (Валютная секция MOEX).
- **Комиссия FORTS (Тариф Стандартный):** `0.45 руб.` брокер за контракт (ИТС) + `~1.00 руб.` биржа = `1.45 руб.` за 1 контракт фьючерса.
- **Исполнение поставочного опциона (двойная ставка ИТС):** `0.45 ₽ * 2 = 0.90 руб.` за контракт.
- **Распределение капитала:** 75% депозита на покупку спот-юаня, 25% остается в рублях на ГО под шорт фьючерса.

In [ ]:
# 1. Установка и импорт необходимых библиотек
!pip install pandas numpy requests matplotlib seaborn

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import time
from datetime import datetime

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Импорт библиотек завершен успешно!')

## 🏛️ 2. Таблица Изменений Ключевой Ставки ЦБ РФ (2023–2026 гг.)
Ключевая ставка Банка России с учетом снижения до 14.0% на 20.08.2026 г.

In [ ]:
cbr_rates_history = [
    ('01.01.2023 - 23.07.2023', 7.50, 'Начало года, стабильная ставка'),
    ('24.07.2023 - 14.08.2023', 8.50, 'Первое повышение ставки'),
    ('15.08.2023 - 17.09.2023', 12.00, 'Внеочередное заседание ЦБ'),
    ('18.09.2023 - 29.10.2023', 13.00, 'Ужесточение ДКП'),
    ('30.10.2023 - 17.12.2023', 15.00, 'Рост инфляционного давления'),
    ('18.12.2023 - 28.07.2024', 16.00, 'Удержание высокой ставки 16%'),
    ('29.07.2024 - 15.09.2024', 18.00, 'Новый этап повышения'),
    ('16.09.2024 - 27.10.2024', 19.00, 'Борьба с перегревом экономики'),
    ('28.10.2024 - 30.06.2025', 21.00, 'Пиковый уровень ставки 21%'),
    ('01.07.2025 - 31.12.2025', 18.00, 'Первое снижение ставки ЦБ'),
    ('01.01.2026 - 31.05.2026', 16.00, 'Постепенная нормализация ДКП'),
    ('01.06.2026 - 20.08.2026', 14.00, 'Текущий уровень ключевой ставки ЦБ (14.0%)')
]

df_cbr = pd.DataFrame(cbr_rates_history, columns=['Период', 'Ключевая ставка ЦБ (%)', 'Комментарий'])
display(df_cbr)

## 📥 3. Загрузка реальных исторических данных MOEX ISS API и Симуляция Бэктеста (15 циклов)

In [ ]:
cbr_rates_tuples = [
    ('2023-01-01', '2023-07-23', 7.50),
    ('2023-07-24', '2023-08-14', 8.50),
    ('2023-08-15', '2023-09-17', 12.00),
    ('2023-09-18', '2023-10-29', 13.00),
    ('2023-10-30', '2023-12-17', 15.00),
    ('2023-12-18', '2024-07-28', 16.00),
    ('2024-07-29', '2024-09-15', 18.00),
    ('2024-09-16', '2024-10-27', 19.00),
    ('2024-10-28', '2025-06-30', 21.00),
    ('2025-07-01', '2025-12-31', 18.00),
    ('2026-01-01', '2026-05-31', 16.00),
    ('2026-06-01', '2026-12-31', 14.00),
]

def get_cbr_rate(date_str):
    for start_d, end_d, rate in cbr_rates_tuples:
        if start_d <= date_str <= end_d:
            return rate
    return 14.00

def fetch_moex_history(engine, market, board, security, date_from, date_till):
    url = f"https://iss.moex.com/iss/history/engines/{engine}/markets/{market}/securities/{security}.json?from={date_from}&till={date_till}"
    if board:
        url = f"https://iss.moex.com/iss/history/engines/{engine}/markets/{market}/boards/{board}/securities/{security}.json?from={date_from}&till={date_till}"
    rows = []
    start = 0
    while True:
        req_url = f"{url}&start={start}"
        r = requests.get(req_url).json()
        data = r['history']['data']
        cols = r['history']['columns']
        if not data:
            break
        df = pd.DataFrame(data, columns=cols)
        rows.append(df)
        start += len(data)
        if len(data) < 100:
            break
    if not rows:
        return pd.DataFrame()
    res = pd.concat(rows, ignore_index=True)
    res['TRADEDATE'] = pd.to_datetime(res['TRADEDATE'])
    return res

print('Загрузка спот-цен CNYRUB_TOM с сервера MOEX...')
spot_df = fetch_moex_history('currency', 'selt', 'CETS', 'CNYRUB_TOM', '2023-01-01', '2026-08-20')
spot_dict = dict(zip(spot_df['TRADEDATE'].dt.strftime('%Y-%m-%d'), spot_df['CLOSE']))

futures_list = [
    ('CRH3', 'CNY-3.23', '2023-01-03', '2023-03-16'),
    ('CRM3', 'CNY-6.23', '2023-03-17', '2023-06-15'),
    ('CRU3', 'CNY-9.23', '2023-06-16', '2023-09-21'),
    ('CRZ3', 'CNY-12.23', '2023-09-22', '2023-12-21'),
    ('CRH4', 'CNY-3.24', '2023-12-22', '2024-03-21'),
    ('CRM4', 'CNY-6.24', '2024-03-22', '2024-06-20'),
    ('CRU4', 'CNY-9.24', '2024-06-21', '2024-09-19'),
    ('CRZ4', 'CNY-12.24', '2024-09-20', '2024-12-19'),
    ('CRH5', 'CNY-3.25', '2024-12-20', '2025-03-20'),
    ('CRM5', 'CNY-6.25', '2025-03-21', '2025-06-19'),
    ('CRU5', 'CNY-9.25', '2025-06-20', '2025-09-18'),
    ('CRZ5', 'CNY-12.25', '2025-09-19', '2025-12-18'),
    ('CRH6', 'CNY-3.26', '2025-12-19', '2026-03-19'),
    ('CRM6', 'CNY-6.26', '2026-03-20', '2026-06-18'),
    ('CRU6', 'CNY-9.26', '2026-06-19', '2026-08-19')
]

initial_capital = 1000000.0
capital = initial_capital
spot_fee_rate = 0.00025  # 0.025%
forts_fee_per_contract = 1.45 # Тариф Стандартный ФОРТС: 0.45 руб. брокер (ИТС) + 1.00 руб. биржа

trade_log = []
equity_curve = [{'date': '2023-01-01', 'capital': initial_capital, 'cbr_rate': 7.50}]

for secid, name, start_d, end_d in futures_list:
    fut_df = fetch_moex_history('futures', 'forts', None, secid, start_d, end_d)
    if fut_df.empty:
        continue
    fut_dict = dict(zip(fut_df['TRADEDATE'].dt.strftime('%Y-%m-%d'), fut_df['CLOSE']))
    
    common_dates = sorted(list(set(fut_dict.keys()).intersection(set(spot_dict.keys()))))
    if not common_dates:
        continue
    entry_date = common_dates[0]
    exit_date = common_dates[-1]
        
    p_spot_entry = spot_dict[entry_date]
    p_fut_entry = fut_dict[entry_date]
    p_spot_exit = spot_dict[exit_date]
    p_fut_exit = fut_dict[exit_date]
    
    days_held = (pd.to_datetime(exit_date) - pd.to_datetime(entry_date)).days
    cbr_rate_period = get_cbr_rate(entry_date)
    
    contango_pct = ((p_fut_entry / p_spot_entry) - 1) * 100
    ann_contango_pct = contango_pct * (365.0 / max(days_held, 1))
    
    if contango_pct > 0.5:
        action = "CASH & CARRY"
        lots = int((capital * 0.75) / (p_spot_entry * 1000))
        cny_amount = lots * 1000
        
        spot_buy_cost = cny_amount * p_spot_entry
        spot_fee_entry = spot_buy_cost * spot_fee_rate
        forts_fee_entry = lots * forts_fee_per_contract
        cash_left = capital - spot_buy_cost - spot_fee_entry - forts_fee_entry
        
        spot_sell_val = cny_amount * p_spot_exit
        spot_fee_exit = spot_sell_val * spot_fee_rate
        forts_fee_exit = lots * forts_fee_per_contract
        fut_pnl = (p_fut_entry - p_fut_exit) * cny_amount
        
        net_pnl = (spot_sell_val - spot_buy_cost) + fut_pnl - (spot_fee_entry + spot_fee_exit + forts_fee_entry + forts_fee_exit)
        capital_end = cash_left + spot_sell_val - spot_fee_exit - forts_fee_exit + fut_pnl
        fees_paid = spot_fee_entry + spot_fee_exit + forts_fee_entry + forts_fee_exit
    else:
        action = "MONEY MARKET (RUSFAR)"
        lots = 0
        cny_amount = 0
        fees_paid = 0.0
        net_pnl = capital * (cbr_rate_period / 100.0) * (days_held / 365.0)
        capital_end = capital + net_pnl
        
    cycle_return_pct = (net_pnl / capital) * 100
    annual_return_pct = cycle_return_pct * (365.0 / max(days_held, 1))
    
    trade_log.append({
        'Контракт': secid,
        'Название': name,
        'Стратегия': action,
        'Дата входа': entry_date,
        'Дата выхода': exit_date,
        'Дней': days_held,
        'Спот Вход': p_spot_entry,
        'Фьюч Вход': p_fut_entry,
        'Контанго (%)': contango_pct,
        'Чистый PnL (руб)': net_pnl,
        'Доходность (APR %)': annual_return_pct,
        'Капитал (руб)': capital_end,
        'Ставка ЦБ (%)': cbr_rate_period
    })
    
    capital = capital_end
    equity_curve.append({'date': exit_date, 'capital': capital, 'cbr_rate': cbr_rate_period})

df_trades = pd.DataFrame(trade_log)
df_equity = pd.DataFrame(equity_curve)
df_equity['date'] = pd.to_datetime(df_equity['date'])

print('=== ИТОГОВАЯ ТАБЛИЦА СДЕЛОК C 01.01.2023 ПО 20.08.2026 (ТАРИФ ФОРТС 0.45 ₽, СТАВКА ЦБ 14%) ===')
display(df_trades[['Контракт', 'Стратегия', 'Дата входа', 'Дата выхода', 'Дней', 'Спот Вход', 'Фьюч Вход', 'Контанго (%)', 'Чистый PnL (руб)', 'Доходность (APR %)', 'Капитал (руб)', 'Ставка ЦБ (%)']])

## 📈 4. Визуализация Динамики Капитала и Ставки ЦБ РФ

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(df_equity['date'], df_equity['capital'], marker='o', color='#1A365D', linewidth=2.5, label='Капитал (руб.)')
ax1.set_title('Динамика Капитала (1,000,000 руб.) на Валютном Cash-and-Carry MOEX (01.01.2023 - 20.08.2026)', fontsize=13, fontweight='bold', pad=12)
ax1.set_ylabel('Капитал (рубли)', fontsize=11)
ax1.yaxis.set_major_formatter('{x:,.0f}')
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.step(df_equity['date'], df_equity['cbr_rate'], where='post', color='#C53030', linewidth=2, label='Ключевая ставка ЦБ РФ (%)')
ax2.set_ylabel('Ставка ЦБ (%)', fontsize=11)
ax2.set_xlabel('Дата', fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

tot_profit = capital - initial_capital
tot_ret = (tot_profit / initial_capital) * 100
print(f"💰 Начальный капитал: 1,000,000.00 руб.")
print(f"🏁 Конечный капитал (на 20.08.2026): {capital:,.2f} руб.")
print(f"📈 Абсолютная чистая прибыль: +{tot_profit:,.2f} руб. (+{tot_ret:.2f}%)")